# Generate Training Dataset

In [149]:
from tqdm.auto import tqdm
import re
import openai
openai.api_key_path = "/Users/spangher/.openai-isi-key.txt"

In [150]:
import pandas as pd 
new_city_council_data = pd.read_json(
    '../data/city-council-meeting-minutes/city-council-processed.jsonl.gz',
    lines=True,
    compression='gzip',
)

new_city_council_data= (
    new_city_council_data
        .assign(data=lambda df: df.apply(lambda x: x['data'] if not isinstance(x['data'], float) else x['pdf_data'], axis=1))
        .drop('pdf_data', axis=1)
)

In [156]:
import pyperclip

pyperclip.copy('\n'.join(new_city_council_data.iloc[-1]['data']))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [36]:
tqdm.pandas()
new_city_council_data['tokens'] = (
    new_city_council_data['data']
    .str.join('\n')
    .progress_apply(lambda x: tokenizer.encode(x))
)

  0%|          | 0/31892 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1264 > 512). Running this sequence through the model will result in indexing errors


In [42]:
training_text = (
    new_city_council_data['tokens']
    .loc[lambda s: s.str.len() > 500]
    .apply(lambda x: x[:3_000])
    .iloc[:5_000]
    .apply(tokenizer.decode)
)

In [108]:
def query_openai_city_council_meeting_minutes(text):
    prompt =f"""Here is the text of a recent city council meeting minute: 

          "{text}"

          -----
          What were the issues discussed? Be descriptive. Return as a bulleted list, for example:
              "
              - Roll call - the attendance of the members of the Alameda Reuse and Redevelopment Authority is recorded.
              - Consent Calendar - various items are approved by the Board without discussion, including the approval of meeting minutes, sublease for sustainable technologies, waiver of license fee for student activities, and an authorization to amend a consultant agreement.
              - Regular Agenda Items - Friends of the Wildlife Refuge present an update on their activities at the Alameda proposed Wildlife Refuge to ensure resources are protected during transfer negotiations. The Board discusses the vulnerability of the Alameda Point and breakwater, increase in pelican numbers, and use of willows by migratory birds. The importance of the Alameda Wildlife Refuge is highlighted, with students from Los Angeles and New York conducting their graduate work at the site.
              "

    """
    
    num_tokens = len(tokenizer.encode(prompt))
    if num_tokens > 4000:
        return 
    
    completion = openai.Completion.create(
      model="text-davinci-003",
      prompt=prompt,
      max_tokens=4056 - num_tokens
    )
    return completion.to_dict_recursive()['choices'][0]['text']

In [ ]:
gpt3_summaries = []

In [126]:
import time

In [ ]:
while True:
    try:
        for text in tqdm(training_text.iloc[len(gpt3_summaries):]):
            summary = query_openai_city_council_meeting_minutes(text)
            gpt3_summaries.append(summary)
    except:
        print('failed, retrying...')
        time.sleep(10)

In [145]:
gpt3_summary_data_so_far = (training_text
 .to_frame('text')
 .iloc[:len(gpt3_summaries)]
 .assign(summary=gpt3_summaries)
 .assign(summary=lambda df: df['summary'].str.strip())
)

In [146]:
gpt3_summary_data_so_far.to_csv('../data/city-council-meeting-minutes/new-agenda-watch-with-gpt3-summaries.csv')

In [147]:
pd.concat([
    pd.read_csv('../data/city-council-meeting-minutes/old-agenda-watch-with-gpt3-summaries-parsed.csv', index_col=0),
    gpt3_summary_data_so_far
])['summary'].isnull().value_counts()

False    3250
True       46
Name: summary, dtype: int64

In [148]:
pd.concat([
    pd.read_csv('../data/city-council-meeting-minutes/old-agenda-watch-with-gpt3-summaries-parsed.csv', index_col=0),
    gpt3_summary_data_so_far
]).dropna().to_json('../modeling/summarization/city-council-summaries.jsonl', orient='records', lines=True)

In [ ]:
from transformers import AutoTokenizer

checkpoint = "t5-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [26]:
from datasets import load_dataset
billsum = load_dataset("billsum")

Found cached dataset billsum (/Users/spangher/.cache/huggingface/datasets/billsum/default/3.0.0/75cf1719d38d6553aa0e0714c393c74579b083ae6e164b2543684e3e92e0c4cc)


  0%|          | 0/3 [00:00<?, ?it/s]

In [32]:
billsum['train']

Dataset({
    features: ['text', 'summary', 'title'],
    num_rows: 18949
})

In [4]:
billsum['text'][0]

'The people of the State of California do enact as follows:\n\n\nSECTION 1.\nThe Legislature finds and declares all of the following:\n(a) (1) Since 1899 congressionally chartered veterans’ organizations have provided a valuable service to our nation’s returning service members. These organizations help preserve the memories and incidents of the great hostilities fought by our nation, and preserve and strengthen comradeship among members.\n(2) These veterans’ organizations also own and manage various properties including lodges, posts, and fraternal halls. These properties act as a safe haven where veterans of all ages and their families can gather together to find camaraderie and fellowship, share stories, and seek support from people who understand their unique experiences. This aids in the healing process for these returning veterans, and ensures their health and happiness.\n(b) As a result of congressional chartering of these veterans’ organizations, the United States Internal Reve

In [7]:
print(billsum['summary'][0])

Existing property tax law establishes a veterans’ organization exemption under which property is exempt from taxation if, among other things, that property is used exclusively for charitable purposes and is owned by a veterans’ organization.
This bill would provide that the veterans’ organization exemption shall not be denied to a property on the basis that the property is used for fraternal, lodge, or social club purposes, and would make specific findings and declarations in that regard. The bill would also provide that the exemption shall not apply to any portion of a property that consists of a bar where alcoholic beverages are served.
Section 2229 of the Revenue and Taxation Code requires the Legislature to reimburse local agencies annually for certain property tax revenues lost as a result of any exemption or classification of property for purposes of ad valorem property taxation.
This bill would provide that, notwithstanding Section 2229 of the Revenue and Taxation Code, no appro

In [11]:
prefix = "summarize: "


def preprocess_function(examples):
    inputs = [prefix + doc for doc in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)
    
    labels = tokenizer(text_target=examples["summary"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [12]:
tokenized_billsum = billsum.map(preprocess_function, batched=True)

Map:   0%|          | 0/1237 [00:00<?, ? examples/s]

In [13]:
tokenized_billsum

Dataset({
    features: ['text', 'summary', 'title', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 1237
})

In [ ]:
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=checkpoint)

In [215]:
from transformers import AutoModel

model = AutoModel.from_pretrained('google/long-t5-local-base')

Some weights of the model checkpoint at google/long-t5-local-base were not used when initializing LongT5Model: ['lm_head.weight']
- This IS expected if you are initializing LongT5Model from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing LongT5Model from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [216]:
len(model.encoder.block)

12

In [217]:
len(model.decoder.block)

12

In [164]:
from transformers import pipeline

In [200]:
summarizer = pipeline("summarization", model="alex2awesome/city_council_gpt3_silver_standard_summaries__t5-large")

In [201]:
test_text = '\n'.join(new_city_council_data['data'].iloc[-2])

In [202]:
tokens = summarizer.tokenizer.encode('Summarize: ' + test_text)

Token indices sequence length is longer than the specified maximum sequence length for this model (1894 > 512). Running this sequence through the model will result in indexing errors


In [203]:
input_text = summarizer.tokenizer.decode( tokens[:500])

In [204]:
summarizer(input_text)

/Users/spangher/opt/anaconda3/lib/python3.9/site-packages/transformers/generation/utils.py:1186: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation)
  warnings.warn(


[{'summary_text': 'this will be a teleconference meeting without a physical location to help stop the spread of COVID-19 . members of the public wishing comment on an item on the agenda may do so in the following ways: 1) e-mail comments by 5:00p.m. on Wednesday, April 6to the Commission at librarycommission@cupertino.org . comments may be made during the public comment period for each agenda item . unregistered participants will be called on by the last four digits'}]

In [213]:
pyperclip.copy('\n'.join(new_city_council_data.iloc[-3]['data']))

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
